## Choose a large transformer model and try to fine-tune it on the dataset of your choice using LoRA.

In [19]:
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer,  AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model
import torch
import numpy as np

In [20]:
news_ds = load_dataset("mdonigian/iab-news-classification")

KeyboardInterrupt: 

In [ ]:
news_ds = news_ds['train'].train_test_split(test_size=0.1)

In [ ]:
news_ds

DatasetDict({
    train: Dataset({
        features: ['url', 'domain', 'maintext', 'iab_category'],
        num_rows: 95652
    })
    test: Dataset({
        features: ['url', 'domain', 'maintext', 'iab_category'],
        num_rows: 10628
    })
})

In [ ]:
train_dataset = news_ds['train']

In [ ]:
train_dataset

Dataset({
    features: ['url', 'domain', 'maintext', 'iab_category'],
    num_rows: 95652
})

In [ ]:
train_dataset, validation_dataset = train_dataset.train_test_split(test_size=0.1).values()

In [ ]:
train_dataset

Dataset({
    features: ['url', 'domain', 'maintext', 'iab_category'],
    num_rows: 86086
})

In [ ]:
news_ds['train'] = train_dataset
news_ds['validation'] = validation_dataset
news_ds

DatasetDict({
    train: Dataset({
        features: ['url', 'domain', 'maintext', 'iab_category'],
        num_rows: 86086
    })
    test: Dataset({
        features: ['url', 'domain', 'maintext', 'iab_category'],
        num_rows: 10628
    })
    validation: Dataset({
        features: ['url', 'domain', 'maintext', 'iab_category'],
        num_rows: 9566
    })
})

In [ ]:
news_ds['train'] = news_ds['train'].rename_column('iab_category', 'labels')
news_ds['test'] = news_ds['test'].rename_column('iab_category', 'labels')
news_ds['validation'] = news_ds['validation'].rename_column('iab_category', 'labels')


In [ ]:
news_ds['train'] = news_ds['train'].class_encode_column('labels')
news_ds['test'] = news_ds['test'].class_encode_column('labels')
news_ds['validation'] = news_ds['validation'].class_encode_column('labels')

Stringifying the column:   0%|          | 0/86086 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/86086 [00:00<?, ? examples/s]

Stringifying the column:   0%|          | 0/10628 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/10628 [00:00<?, ? examples/s]

Stringifying the column:   0%|          | 0/9566 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/9566 [00:00<?, ? examples/s]

In [ ]:
news_ds['train']['labels'][:5]

[29, 25, 6, 8, 1]

In [ ]:
model_id = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=35, device_map='auto')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenize(batch):
    return tokenizer(batch['maintext'], truncation=True, max_length=512)

news_ds= news_ds.map(tokenize, batched=True)

Map:   0%|          | 0/86086 [00:00<?, ? examples/s]

Map:   0%|          | 0/10628 [00:00<?, ? examples/s]

Map:   0%|          | 0/9566 [00:00<?, ? examples/s]

In [ ]:
for name, module in model.named_modules():
    print(name, module)

 DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout

In [ ]:
lora_config = LoraConfig(r= 16,
                         lora_alpha = 32,
                         target_modules=['q_lin', 'v_lin', 'pre_classifier', "classifier"],
                         lora_dropout=0.1,
                         bias='none',
                         task_type="SEQ_CLS")

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def compute_metric(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy" : (preds == labels).mean()}

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer)

training_args = TrainingArguments(
    output_dir = 'lora_bert',
    num_train_epochs=3,
    gradient_accumulation_steps= 4,
    per_device_train_batch_size= 8,
    learning_rate=2e-4,
    logging_steps=250,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True
)

trainer = Trainer(
    model = model,
    args=training_args,
    train_dataset=news_ds['train'],
    eval_dataset=news_ds['validation'],
    processing_class = tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metric
)
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,11.188988,2.714773,0.434664


KeyboardInterrupt: 

In [ ]:
trainer.predict(news_ds['test'])

: 